# Fine-Tuning Laya on `LocalLLaMA/typed-decisions`

[![PyPI version](https://img.shields.io/pypi/v/laya.svg)](https://pypi.org/project/laya/)
[![Hugging Face Model](https://img.shields.io/badge/%F0%9F%A4%97%20Model-convaiinnovations%2Flaya-blue)](https://huggingface.co/convaiinnovations/laya)
[![Dataset](https://img.shields.io/badge/%F0%9F%A4%97%20Dataset-LocalLLaMA%2Ftyped--decisions-green)](https://huggingface.co/datasets/LocalLLaMA/typed-decisions)

This notebook fine-tunes **Laya** (`convaiinnovations/laya`, 421M params) on the **1,200 training cases (6,000 typed decisions)** of the [LocalLLaMA/typed-decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions) benchmark, then evaluates the result against the official test split.

It runs on whatever this machine provides:

| host | how it trains |
|---|---|
| 2+ NVIDIA T4s (Kaggle) or any multi-GPU CUDA box | Distributed Data Parallel via `torchrun`, fp16 autocast + loss scaling |
| an **AMD GPU through Metal (MPS)** on macOS | single process, fp32 |
| a **ROCm** build of PyTorch (Linux, AMD) | ROCm presents itself as `cuda`, so the multi-GPU path applies unchanged |
| CPU | single process, fp32 — slow, but exact |

The loop itself lives in `laya.finetune`; this notebook prepares the data, runs it, evaluates, and optionally publishes. Mixed precision is CUDA-only: MPS and CPU run fp32 with no gradient scaler, because torch has no MPS autocast backend and there is nothing to scale in fp32.

---

### Settings

* **Kaggle:** Notebook options → **Accelerator `GPU T4 x2`**, **Internet `On`**. Outputs go to `/kaggle/working/`.
* **Anywhere else:** nothing to configure. Outputs go to `./finetune_run/`, and an existing local checkpoint (`models/laya`) is used instead of re-downloading when one is found.
* **Smoke run** (a few minutes instead of hours):
  `LAYA_FINETUNE_LIMIT=48 LAYA_FINETUNE_EPOCHS=1 LAYA_EVAL_LIMIT=40`
* **Force a device:** `LAYA_DEVICE=mps` (or `cpu`, `cuda`).


## 1. Environment & device check

Report what this machine can train on, and what the training loop will therefore do: DDP across several GPUs, or a single process on MPS or CPU.


In [ ]:
import os, platform, torch

devices = []
N_CUDA = torch.cuda.device_count()
if torch.cuda.is_available():
    backend = "ROCm" if getattr(torch.version, "hip", None) else "CUDA"
    for i in range(N_CUDA):
        p = torch.cuda.get_device_properties(i)
        devices.append(f"{backend} {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    devices.append("MPS: Metal GPU (AMD or Apple silicon)")
if not devices:
    devices.append("CPU only")

USE_DDP = N_CUDA >= 2
if USE_DDP:
    PLAN = f"DDP across {N_CUDA} GPUs via torchrun, fp16 autocast"
elif N_CUDA == 1:
    PLAN = "single process on cuda, fp16 autocast"
elif any(d.startswith("MPS") for d in devices):
    PLAN = "single process on mps, fp32 (no autocast, no loss scaler)"
else:
    PLAN = "single process on cpu, fp32"

print("PyTorch %s | python %s" % (torch.__version__, platform.python_version()))
print("Platform      :", platform.platform())
print("Training devices available:")
for d in devices:
    print("  -", d)
print()
print("Plan          :", PLAN)
print()
print("Notes:")
print("  * MPS has no autocast backend in torch 2.x, so it trains in fp32. That is slower per")
print("    step but numerically safe; the reward, the policy gradient and the calibration are")
print("    the same code.")
print("  * A ROCm build reports itself as 'cuda', so the multi-GPU path above applies to AMD")
print("    cards on Linux without changes.")

## 2. Install dependencies

Only what is missing is installed, so a prepared environment is left alone.


In [ ]:
import importlib.util, subprocess, sys

REQUIRED = ["laya", "transformers", "datasets", "safetensors", "huggingface_hub", "tabulate"]
missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]
if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *missing], check=False)
else:
    print("Dependencies already present; skipping install.")

import datasets, laya, transformers, torch

print("Laya version        :", laya.__version__)
print("Transformers version:", transformers.__version__)
print("PyTorch version     :", torch.__version__)
print("Datasets version    :", datasets.__version__)

if tuple(int(x) for x in transformers.__version__.split(".")[:2]) < (4, 48):
    print("\nWARNING: transformers >= 4.48 is needed to load the ModernBERT/mmBERT encoders.")

## 3. Download & preprocess the data

Every case is tokenized once into items on disk, so any number of training ranks can read the same file. `LAYA_FINETUNE_LIMIT` caps the item count for a smoke run.


In [ ]:
import os, json, torch
from datasets import load_dataset
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, render_options, QTYPES

# Kaggle writes to /kaggle/working; anywhere else use ./finetune_run next to the notebook.
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("finetune_run")
os.makedirs(WORK_DIR, exist_ok=True)
LIMIT = int(os.environ.get("LAYA_FINETUNE_LIMIT", "0"))
print("Working directory:", WORK_DIR, "| item limit:", LIMIT or "none")

MODEL_ID = "convaiinnovations/laya"
# Prefer a checkpoint that is already on disk (models/laya in a checkout) over re-downloading.
CANDIDATES = [os.environ.get("LAYA_MODEL_DIR"), "models/laya", "../models/laya"]
model_dir = next((os.path.abspath(c) for c in CANDIDATES
                  if c and os.path.exists(os.path.join(c, "model.safetensors"))), None)
if model_dir is None:
    print(f"Fetching tokenizer and config from {MODEL_ID}...")
    model_dir = snapshot_download(MODEL_ID)
else:
    print("Using local checkpoint:", model_dir)
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)

print("Downloading LocalLLaMA/typed-decisions (train split)...")
ds_train = load_dataset("LocalLLaMA/typed-decisions", "all", split="train")

def build_training_item(state, q, gold_q):
    t = q["type"]
    crit = q.get("criteria", {})
    if t == "choice":
        keys = list(crit.keys())
        target = [gold_q["probabilities"].get(k, 0.0) for k in keys]
    elif t == "noul":
        target = [gold_q["probabilities"].get("false", 0.5), gold_q["probabilities"].get("true", 0.5)]
    elif t == "score":
        n_levels = len(crit) if isinstance(crit, list) else 4
        target = [gold_q["probabilities"].get(str(i), 0.0) for i in range(n_levels)]

    s = sum(target)
    target = [v / s for v in target] if s > 0 else [1.0 / len(target)] * len(target)
    label = target.index(max(target))
    k = len(render_options({"t": t, "crit": crit}))

    seq, markers = build_sequence(tok, state, {"t": t, "ins": q["instructions"], "crit": crit},
                                  cfg["max_len"], cfg["head_max_len"])
    if len(markers) != k:
        return None
    return {"ids": seq, "markers": markers, "qtype": QTYPES[t], "target": target, "label": label}

items = []
for row in ds_train:
    state = json.loads(row["state"])
    questions = json.loads(row["questions"])
    gold = json.loads(row["gold"])
    for qid, q in questions.items():
        if qid in gold:
            it = build_training_item(state, q, gold[qid])
            if it:
                items.append(it)

if LIMIT:
    items = items[:LIMIT]

ITEMS_PATH = os.path.join(WORK_DIR, "train_items.json")
with open(ITEMS_PATH, "w") as f:
    json.dump(items, f)
print(f"Preprocessed {len(items)} training sequences across {len(ds_train)} cases.")
print("Saved items to", ITEMS_PATH)

## 4. The training loop

`laya.finetune.train_rlcd` is the RLCD loop: a strictly proper scoring reward (log + spherical, plus a ranked probability term for ordinal questions) evaluated over a group of noisy samples of the logits, a policy-gradient update on the advantage, a soft cross-entropy term pulling towards the teacher distribution, then post-training temperature calibration. The same function runs DDP on CUDA/ROCm and single-process on MPS or CPU.


In [ ]:
import os
from laya.finetune import device_report, pick_device, train_rlcd

device = pick_device(os.environ.get("LAYA_DEVICE"))
EPOCHS = int(os.environ.get("LAYA_FINETUNE_EPOCHS", "4"))
MICRO_BATCH = int(os.environ.get("LAYA_FINETUNE_MICRO_BATCH", "8"))
GRAD_ACCUM = int(os.environ.get("LAYA_FINETUNE_GRAD_ACCUM", "4"))
GROUP_SIZE = 4
MAX_LEN = int(os.environ.get("LAYA_FINETUNE_MAX_LEN", "1024"))
HEAD_MAX_LEN = int(os.environ.get("LAYA_FINETUNE_HEAD_MAX_LEN", "256"))

print(device_report())
print("Training device :", device)
print("Schedule        : %d epochs | micro-batch %d x accum %d | group size %d"
      % (EPOCHS, MICRO_BATCH, GRAD_ACCUM, GROUP_SIZE))
print("Exported context: max_len %d, head_max_len %d" % (MAX_LEN, HEAD_MAX_LEN))
if device.type != "cuda":
    print("Precision       : fp32 (autocast and loss scaling are CUDA-only)")

## 5. Fine-tune

On 2+ GPUs this launches `torchrun` so the ranks share the work; on one GPU, MPS or CPU it runs in-process. A full 4-epoch run over ~6,000 items is hours on a T4 pair and considerably longer on one desktop GPU, so use the smoke-run variables when you are checking the plumbing.


In [ ]:
import subprocess, sys

OUTPUT_DIR = os.path.join(WORK_DIR, "laya_finetuned_typed_decisions")
common = ["--model-dir", model_dir, "--items", ITEMS_PATH, "--output-dir", OUTPUT_DIR,
          "--epochs", str(EPOCHS), "--micro-batch", str(MICRO_BATCH),
          "--grad-accum", str(GRAD_ACCUM), "--group-size", str(GROUP_SIZE),
          "--max-len", str(MAX_LEN), "--head-max-len", str(HEAD_MAX_LEN)]

if USE_DDP:
    cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone",
           f"--nproc_per_node={N_CUDA}", "-m", "laya.finetune", *common]
    print("Launching:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    metrics = train_rlcd(items, model_dir, OUTPUT_DIR, device=device, epochs=EPOCHS,
                         micro_batch=MICRO_BATCH, grad_accum=GRAD_ACCUM, group_size=GROUP_SIZE,
                         max_len=MAX_LEN, head_max_len=HEAD_MAX_LEN)
    print(json.dumps(metrics, indent=2) if "json" in dir() else metrics)

print("\nSaved to", OUTPUT_DIR)

## 6. Benchmark evaluation on the test split

Evaluate the fine-tuned checkpoint against the official `test` split, on the same device that trained it (or `cpu` via `LAYA_DEVICE`). `LAYA_EVAL_LIMIT` caps the number of cases for a smoke run.


In [ ]:
import json, os, time
import numpy as np
import pandas as pd
from datasets import load_dataset
import laya
from laya.common import ece_score

EVAL_LIMIT = int(os.environ.get("LAYA_EVAL_LIMIT", "0"))
print("Loading test split for evaluation...")
ds_test = load_dataset("LocalLLaMA/typed-decisions", "all", split="test")
if EVAL_LIMIT:
    ds_test = ds_test.select(range(min(EVAL_LIMIT, len(ds_test))))

agent_ft = laya.Agent(OUTPUT_DIR, device=str(device))
print(f"Evaluating {len(ds_test)} test cases on {agent_ft.device}...")

predictions = []
latencies_ms = []
t0_eval = time.time()

for i, row in enumerate(ds_test):
    case_id = row["id"]
    workflow = row["workflow"]
    state = json.loads(row["state"])
    questions = json.loads(row["questions"])
    gold = json.loads(row["gold"])

    t0 = time.perf_counter()
    res = agent_ft.predict(state, questions)
    dt_ms = (time.perf_counter() - t0) * 1000
    latencies_ms.append(dt_ms)

    predictions.append({
        "id": case_id,
        "workflow": workflow,
        "pred": res["answers"],
        "gold": gold,
        "questions": questions,
        "latency_ms": dt_ms
    })

print(f"Evaluated all {len(ds_test)} cases in {time.time() - t0_eval:.1f}s!")

## 7. Compute Official Metrics & Comparison Table against TypeSafe Jev


In [ ]:
accuracies = []
soft_accuracies = []
brier_scores = []
kl_divs = []
tv_distances = []
score_maes = []
within_one = []
all_confs = []
all_corrects = []

for item in predictions:
    pred_answers = item["pred"]
    gold_answers = item["gold"]
    questions = item["questions"]
    
    for qid, qdef in questions.items():
        p_ans = pred_answers[qid]
        g_ans = gold_answers[qid]
        q_type = qdef["type"]
        
        # 1. Choice
        if q_type == "choice":
            keys = list(qdef["criteria"].keys())
            pred_choice = p_ans["choice"]
            gold_label = str(g_ans["label"])
            
            is_corr = float(pred_choice == gold_label)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)
            
            p_probs = np.array([p_ans["probabilities"].get(k, 1e-6) for k in keys])
            g_probs = np.array([g_ans["probabilities"].get(k, 1e-6) for k in keys])
            p_probs /= p_probs.sum()
            g_probs /= g_probs.sum()
            
            all_confs.append(float(p_probs.max()))
            soft_accuracies.append(float((p_probs * g_probs).sum()))
            brier_scores.append(float(((p_probs - g_probs) ** 2).sum()))
            tv_distances.append(float(0.5 * np.abs(p_probs - g_probs).sum()))
            kl_divs.append(float((g_probs * np.log(np.clip(g_probs / p_probs, 1e-12, 1e4))).sum()))
            
        # 2. Noul
        elif q_type == "noul":
            p_val = p_ans["noul"]
            g_val = g_ans.get("noul", g_ans.get("probabilities", {}).get("true", 0.5))
            gold_label = str(g_ans["label"]).lower()
            
            pred_label = "true" if p_val >= 0.5 else "false"
            is_corr = float(pred_label == gold_label)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)
            all_confs.append(float(max(p_val, 1.0 - p_val)))
            
            p_dist = np.array([1.0 - p_val, p_val])
            g_dist = np.array([1.0 - g_val, g_val])
            
            soft_accuracies.append(float((p_dist * g_dist).sum()))
            brier_scores.append(float(((p_dist - g_dist) ** 2).sum()))
            tv_distances.append(float(0.5 * np.abs(p_dist - g_dist).sum()))
            kl_divs.append(float((g_dist * np.log(np.clip(g_dist / p_dist, 1e-12, 1e4))).sum()))
            
        # 3. Score
        elif q_type == "score":
            p_score = p_ans["score"]
            g_score = g_ans.get("score", 0.0)
            score_maes.append(abs(p_score - g_score))
            within_one.append(float(abs(p_score - g_score) <= 1.0))
            
            # Use argmax of probabilities for discrete prediction
            n_levels = len(qdef.get("criteria", []))
            p_probs_score = np.array([p_ans["probabilities"].get(str(i), 0.0) for i in range(n_levels)])
            if p_probs_score.sum() > 0:
                p_probs_score /= p_probs_score.sum()
                p_lvl = int(np.argmax(p_probs_score))
                all_confs.append(float(p_probs_score.max()))
            else:
                p_lvl = int(round(p_score))
                all_confs.append(0.5)
                
            g_lvl = int(g_ans.get("label", int(round(g_score))))
            is_corr = float(p_lvl == g_lvl)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)

laya_acc = np.mean(accuracies)
laya_soft_acc = np.mean(soft_accuracies)
laya_brier = np.mean(brier_scores)
laya_kl = np.mean(kl_divs)
laya_tv = np.mean(tv_distances)
laya_ece = ece_score(np.array(all_confs), np.array(all_corrects))
laya_mae = np.mean(score_maes) if score_maes else 0.0
laya_within1 = np.mean(within_one) if within_one else 0.0
laya_latency = np.percentile(latencies_ms, 50)

comparison_data = [
    {
        "Model": "TypeSafe Jev 1.13.0",
        "Kind": "general",
        "Accuracy": 0.727,
        "Soft Acc": 0.580,
        "Brier": 0.148,
        "ECE": 0.144,
        "Score MAE": 0.391,
        "Within 1 Level": "0.952",
        "ms/case": 710,
        "Cost/Case": "$0.0004 (API)"
    },
    {
        "Model": "Laya (Fine-Tuned, %s)" % agent_ft.device,
        "Kind": "fine-tuned",
        "Accuracy": round(laya_acc, 3),
        "Soft Acc": round(laya_soft_acc, 3),
        "Brier": round(laya_brier, 3),
        "ECE": round(laya_ece, 3),
        "Score MAE": round(laya_mae, 3),
        "Within 1 Level": f"{laya_within1:.3f}",
        "ms/case": round(laya_latency, 1),
        "Cost/Case": "$0.00 (Self-Hosted)"
    },
    {
        "Model": "ModernBERT-base (149M)",
        "Kind": "specialist",
        "Accuracy": 0.646,
        "Soft Acc": 0.542,
        "Brier": 0.119,
        "ECE": 0.179,
        "Score MAE": 0.444,
        "Within 1 Level": "0.931",
        "ms/case": 349,
        "Cost/Case": "$0.00"
    },
    {
        "Model": "Teacher Self-Agreement",
        "Kind": "ceiling",
        "Accuracy": 0.735,
        "Soft Acc": "-",
        "Brier": "-",
        "ECE": "-",
        "Score MAE": "-",
        "Within 1 Level": "-",
        "ms/case": "-",
        "Cost/Case": "-"
    }
]

df_comp = pd.DataFrame(comparison_data)
print("=== HEAD-TO-HEAD BENCHMARK TABLE ===\n")
print(df_comp.to_markdown(index=False))


## 8. (Optional) Push to Hugging Face Hub


In [ ]:
import os
from huggingface_hub import HfApi

# Optional cell. Set HF_TOKEN (on Kaggle: Add-ons -> Secrets -> HF_TOKEN) to publish; without one
# it writes the model card locally and stops, so a local or smoke run is never broken here.
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")

NEW_REPO = os.environ.get("LAYA_PUSH_REPO", "convaiinnovations/laya-typed-decisions")
print("Model card target: %s%s" % (NEW_REPO, "" if token else "  (no HF_TOKEN -- will not upload)"))

# Reference baselines pulled from the comparison table above, so the model
# card's narrative always matches the numbers actually computed this run.
JEV_ACC = next(r["Accuracy"] for r in comparison_data if r["Model"] == "TypeSafe Jev 1.13.0")
CEILING_ACC = next(r["Accuracy"] for r in comparison_data if r["Model"] == "Teacher Self-Agreement")

def describe_vs_baseline(acc, jev_acc=JEV_ACC, ceiling_acc=CEILING_ACC):
    """Builds an accuracy comparison sentence that reflects this run's actual result,
    instead of assuming the fine-tuned model always wins."""
    if acc > ceiling_acc:
        return (f"outperforming **TypeSafe Jev 1.13.0 ({jev_acc:.3f})** and surpassing the "
                f"benchmark's **Teacher Self-Agreement ceiling ({ceiling_acc:.3f})**")
    elif acc > jev_acc:
        return (f"outperforming **TypeSafe Jev 1.13.0 ({jev_acc:.3f})**, though still below the "
                f"benchmark's Teacher Self-Agreement ceiling ({ceiling_acc:.3f})")
    else:
        return (f"trailing **TypeSafe Jev 1.13.0 ({jev_acc:.3f})** and the benchmark's "
                f"Teacher Self-Agreement ceiling ({ceiling_acc:.3f})")

comparison_sentence = describe_vs_baseline(laya_acc)

# 1. Write comprehensive model card README.md into OUTPUT_DIR before upload
readme_content = f"""---
license: apache-2.0
library_name: transformers
tags:
- laya
- system-one
- calibrated-decisions
- rlcd
- structured-decisions
- typed-decisions
- benchmark
metrics:
- accuracy
- brier_score
model-index:
- name: laya-typed-decisions
  results:
  - task:
      type: text-classification
      name: System One Decision Benchmark
    dataset:
      type: LocalLLaMA/typed-decisions
      name: Typed Decisions
    metrics:
    - type: accuracy
      value: {laya_acc:.3f}
    - type: brier_score
      value: {laya_brier:.3f}
---

# Laya (Fine-Tuned on Typed-Decisions Benchmark)

This is **Laya** fine-tuned on the 1,200 training cases (6,000 decisions) of the independent [LocalLLaMA/typed-decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions) benchmark.

On the official 400-case test set (2,000 decisions across Agent Trace Observability, Customer Service, Invoice Processing, and Security Incidents), it achieves **{laya_acc:.3f} Accuracy**, {comparison_sentence}.

## Head-to-Head Benchmark Results

| Model | Kind | Accuracy | Soft Acc | Brier Score | ECE | Score MAE | Within 1 Level | Latency (p50) | Cost/Case |
|---|---|---|---|---|---|---|---|---|---|
| **Laya (Ours)** | **fine-tuned** | **{laya_acc:.3f}** | **{laya_soft_acc:.3f}** | **{laya_brier:.3f}** | **{laya_ece:.3f}** | **{laya_mae:.3f}** | **{laya_within1:.3f}** | **{laya_latency:.1f} ms** | **$0.00 (Self-Hosted)** |
| TypeSafe Jev 1.13.0 | general | 0.727 | 0.580 | 0.148 | 0.144 | 0.391 | 0.952 | 710 ms | $0.0004 (API) |
| ModernBERT-base (149M) | specialist | 0.646 | 0.542 | 0.119 | 0.179 | 0.444 | 0.931 | 349 ms | $0.00 |
| Teacher Self-Agreement | ceiling | 0.735 | - | - | - | - | - | - | - |

## Installation & Quickstart

```bash
pip install laya
```

```python
import laya

# Load the fine-tuned model directly from Hugging Face
agent = laya.load("convaiinnovations/laya-typed-decisions")

# Evaluate any workflow state and typed questions in a single forward pass
result = agent.predict(state, questions)
print(result["answers"])
```

## License
Apache 2.0. Developed by [Convai Innovations](https://huggingface.co/convaiinnovations).
"""

with open(os.path.join(OUTPUT_DIR, "README.md"), "w") as f:
    f.write(readme_content)

print(f"Generated model card at {os.path.join(OUTPUT_DIR, 'README.md')}")

if not token:
    print("No HF_TOKEN available, so nothing was uploaded.")
    print("The model card is ready at", os.path.join(OUTPUT_DIR, "README.md"))
    print("Re-run this cell with a token, or upload manually with `hf upload`.")
else:
    api = HfApi(token=token)
    api.create_repo(NEW_REPO, repo_type="model", private=False, exist_ok=True)

    # Upload the fine-tuned weights, config, tokenizer, and README
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=NEW_REPO,
        repo_type="model",
        commit_message=f"Laya fine-tuned on typed-decisions: Accuracy {laya_acc:.3f} vs Jev {JEV_ACC:.3f} / Ceiling {CEILING_ACC:.3f}"
    )
    print("\nModel successfully published to: https://huggingface.co/%s" % NEW_REPO)


## 9. Save Benchmark Report as JSON
Export the full evaluation metrics, comparison table, and per-workflow accuracy breakdown to JSON.


In [ ]:
import json, os

report = {
    "benchmark": "LocalLLaMA/typed-decisions",
    "model": "Laya (Fine-Tuned, %s)" % agent_ft.device,
    "n_cases": len(ds_test),
    "n_decisions": len(ds_test) * 5,
    "metrics": {
        "accuracy": round(float(laya_acc), 4),
        "soft_accuracy": round(float(laya_soft_acc), 4),
        "brier_score": round(float(laya_brier), 4),
        "ece": round(float(laya_ece), 4),
        "score_mae": round(float(laya_mae), 4),
        "within_1_level": round(float(laya_within1), 4),
        "latency_p50_ms": round(float(laya_latency), 1),
        "latency_p95_ms": round(float(np.percentile(latencies_ms, 95)), 1),
        "kl_divergence": round(float(laya_kl), 4),
        "total_variation": round(float(laya_tv), 4)
    },
    "comparison": comparison_data,
    "per_workflow": {}
}

# Calculate per-workflow accuracy
for wf in ["agent_trace_observability", "customer_service", "invoice_processing", "security_incidents"]:
    wf_items = [item for item in predictions if item["workflow"] == wf]
    wf_corr = []
    for item in wf_items:
        for qid, qdef in item["questions"].items():
            p_ans = item["pred"][qid]
            g_ans = item["gold"][qid]
            if qdef["type"] == "choice":
                wf_corr.append(float(p_ans["choice"] == str(g_ans["label"])))
            elif qdef["type"] == "noul":
                pred_label = "true" if p_ans["noul"] >= 0.5 else "false"
                wf_corr.append(float(pred_label == str(g_ans["label"]).lower()))
            elif qdef["type"] == "score":
                p_probs = [p_ans["probabilities"].get(str(i), 0.0) for i in range(len(qdef.get("criteria", [])))]
                p_lvl = int(np.argmax(p_probs)) if sum(p_probs) > 0 else int(round(p_ans["score"]))
                g_lvl = int(g_ans.get("label", int(round(g_ans.get("score", 0.0)))))
                wf_corr.append(float(p_lvl == g_lvl))
    if wf_corr:
        report["per_workflow"][wf] = {
            "n_decisions": len(wf_corr),
            "accuracy": round(float(np.mean(wf_corr)), 4)
        }

# Save in working directory and model directory
out_file1 = os.path.join(WORK_DIR, "laya_typed_decisions_benchmark_report.json")
out_file2 = os.path.join(OUTPUT_DIR, "benchmark_report.json")

with open(out_file1, "w") as f:
    json.dump(report, f, indent=2)

with open(out_file2, "w") as f:
    json.dump(report, f, indent=2)

print(f"Benchmark results successfully saved to:")
print(f"  - {out_file1}")
print(f"  - {out_file2}")
print("\nSummary:")
print(json.dumps(report["metrics"], indent=2))
print("\nPer-Workflow Accuracy:")
print(json.dumps(report["per_workflow"], indent=2))
